<a href="https://colab.research.google.com/github/almughairy/NGC2099/blob/main/M37_Gaia_DR3_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install astroquery

from astroquery.gaia import Gaia
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:

"""GAIA DR3 DATA (2022)"""
#DATA 3

# 3. Define the approximate centre of M37 / NGC 2099
ra = 88.075      # Right Ascension in degrees
dec = 32.553     # Declination in degrees
radius = 1.0     # Search radius in degrees

# 4. Write the Gaia DR3 query
query_dr3 = f"""
SELECT TOP 150000
    source_id,
    ra,
    dec,
    parallax,
    parallax_error,
    pmra,
    pmra_error,
    pmdec,
    pmdec_error,
    phot_g_mean_mag,
    bp_rp
FROM gaiadr3.gaia_source
WHERE 1 = CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', {ra}, {dec}, {radius})
)
AND parallax IS NOT NULL
AND pmra IS NOT NULL
AND pmdec IS NOT NULL
AND bp_rp IS NOT NULL
"""


# 6. Convert the Gaia table into a pandas dataframe
job_dr3 = Gaia.launch_job_async(query_dr3)
results_dr3 = job_dr3.get_results()
dr3 = results_dr3.to_pandas()

# 7. Show the first few rows
dr3.head()

dr3.info()
dr3.describe()

In [ ]:

"""GAIA EDR3 DATA (2020)"""
#DATA 2

# 3. Define the approximate centre of M37 / NGC 2099
ra = 88.075      # Right Ascension in degrees
dec = 32.553     # Declination in degrees
radius = 1.0     # Search radius in degrees

# 4. Write the Gaia EDR3 query
query_edr3 = f"""
SELECT TOP 150000
    source_id,
    ra,
    dec,
    parallax,
    parallax_error,
    pmra,
    pmra_error,
    pmdec,
    pmdec_error,
    phot_g_mean_mag,
    bp_rp
FROM gaiaedr3.gaia_source
WHERE 1 = CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', {ra}, {dec}, {radius})
)
AND parallax IS NOT NULL
AND pmra IS NOT NULL
AND pmdec IS NOT NULL
AND bp_rp IS NOT NULL
"""


# 6. Convert the Gaia table into a pandas dataframe
job_edr3 = Gaia.launch_job_async(query_edr3)
results_edr3 = job_edr3.get_results()
edr3 = results_edr3.to_pandas()

# 7. Show the first few rows
edr3.head()
edr3.info()
edr3.describe()

In [4]:

"""GAIA DR2 DATA (2018)"""
#DATA 1

# 3. Define the approximate centre of M37 / NGC 2099
ra = 88.075      # Right Ascension in degrees
dec = 32.553     # Declination in degrees
radius = 1.0     # Search radius in degrees

# 4. Write the Gaia EDR3 query
query_dr2 = f"""
SELECT TOP 150000
    source_id,
    ra,
    dec,
    parallax,
    parallax_error,
    pmra,
    pmra_error,
    pmdec,
    pmdec_error,
    phot_g_mean_mag,
    bp_rp
FROM gaiadr2.gaia_source
WHERE 1 = CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', {ra}, {dec}, {radius})
)
AND parallax IS NOT NULL
AND pmra IS NOT NULL
AND pmdec IS NOT NULL
AND bp_rp IS NOT NULL
"""


# 6. Convert the Gaia table into a pandas dataframe
job_dr2 = Gaia.launch_job_async(query_dr2)
results_dr2 = job_dr2.get_results()
dr2 = results_dr2.to_pandas()

# 7. Show the first few rows
dr2.head()
dr2.info()
dr2.describe()

KeyboardInterrupt: 

In [ ]:
#checking data lengths for three data sets

print(len(dr2))
print(len(edr3))
print(len(dr3))

#right angle declenation plot for dr3 data

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)

datasets = [
    (dr2, "Gaia DR2"),
    (edr3, "Gaia EDR3"),
    (dr3, "Gaia DR3")
]

for ax, (data, title) in zip(axes, datasets):
    h = ax.hist2d(
        data["ra"],
        data["dec"],
        bins=250
    )
    ax.set_title(title)
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")

fig.suptitle("Sky Density Around M37", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
ra_min, ra_max = edr3["ra"].min(), edr3["ra"].max()
dec_min, dec_max = edr3["dec"].min(), edr3["dec"].max()

bins = 1500

H_edr3, xedges, yedges = np.histogram2d(
    edr3["ra"], edr3["dec"],
    bins=bins,
    range=[[ra_min, ra_max], [dec_min, dec_max]]
)

H_dr3, _, _ = np.histogram2d(
    dr3["ra"], dr3["dec"],
    bins=[xedges, yedges]
)

diff = H_dr3 - H_edr3

plt.figure(figsize=(8, 6))
plt.imshow(
    diff.T,
    origin="lower",
    extent=[ra_min, ra_max, dec_min, dec_max],
    aspect="auto",
    cmap="coolwarm"
)
plt.colorbar(label="DR3 - EDR3 source count per bin")
plt.xlabel("RA (deg)")
plt.ylabel("Dec (deg)")
plt.title("Source Density Difference: DR3 - EDR3")
plt.gca().invert_xaxis()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define a common RA-Dec range
ra_min, ra_max = min(dr2["ra"].min(), edr3["ra"].min()), max(dr2["ra"].max(), edr3["ra"].max())
dec_min, dec_max = min(dr2["dec"].min(), edr3["dec"].min()), max(dr2["dec"].max(), edr3["dec"].max())

bins = 50

# Create the DR2 density map
H_dr2, xedges, yedges = np.histogram2d(
    dr2["ra"],
    dr2["dec"],
    bins=bins,
    range=[[ra_min, ra_max], [dec_min, dec_max]]
)

# Create the EDR3 density map using EXACTLY the same bins
H_edr3, _, _ = np.histogram2d(
    edr3["ra"],
    edr3["dec"],
    bins=[xedges, yedges]
)

# Difference map
diff = H_edr3 - H_dr2

# Plot
plt.figure(figsize=(8, 6))

plt.imshow(
    diff.T,
    origin="lower",
    extent=[ra_min, ra_max, dec_min, dec_max],
    aspect="auto",
    cmap="coolwarm"
)

plt.colorbar(label="EDR3 - DR2 source count per bin")
plt.xlabel("RA (deg)")
plt.ylabel("Dec (deg)")
plt.title("Source Density Difference: EDR3 - DR2")
#plt.gca().invert_xaxis()

plt.show()

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5),
    sharex=True,
    sharey=True,
    constrained_layout=True
)
datasets = [
    (dr2, "Gaia DR2"),
    (edr3, "Gaia EDR3"),
    (dr3, "Gaia DR3")
]

for ax, (data, title) in zip(axes, datasets):
    h = ax.hist2d(
        data["pmra"],
        data["pmdec"],
        bins=250,
        range=[[-5, 5], [-5, 5]]
    )
    ax.set_title(title)
    ax.set_xlabel("pmRA (mas/yr)")
    ax.set_ylabel("pmDec (mas/yr)")

fig.colorbar(h[3], ax=axes, label="Number of sources per bin")
fig.suptitle("Proper-Motion Density Around M37", fontsize=16)
plt.show()

In [ ]:
# --- First-pass M37 membership selection using DR3 ---
# This is a simple exploratory selection, not the final publishable method.

import numpy as np
import matplotlib.pyplot as plt

# Approximate M37 cluster values from the proper-motion plot
pmra_c = 0.6
pmdec_c = -1.4
parallax_c = 0.67

# Selection widths
pm_radius = 0.65
parallax_width = 0.20

# Select probable members using proper motion + parallax
members_dr3 = dr3[
    ((dr3["pmra"] - pmra_c)**2 + (dr3["pmdec"] - pmdec_c)**2 < pm_radius**2) &
    (dr3["parallax"] > parallax_c - parallax_width) &
    (dr3["parallax"] < parallax_c + parallax_width)
].copy()

print("Selected probable M37 members:", len(members_dr3))

# --- Plot raw vs selected sample ---

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Raw RA-Dec
axes[0, 0].hist2d(dr3["ra"], dr3["dec"], bins=150)
axes[0, 0].set_title("Raw DR3 RA-Dec")
axes[0, 0].set_xlabel("RA (deg)")
axes[0, 0].set_ylabel("Dec (deg)")
axes[0, 0].invert_xaxis()

# Selected RA-Dec
axes[0, 1].hist2d(members_dr3["ra"], members_dr3["dec"], bins=80)
axes[0, 1].set_title("Selected Members RA-Dec")
axes[0, 1].set_xlabel("RA (deg)")
axes[0, 1].set_ylabel("Dec (deg)")
axes[0, 1].invert_xaxis()

# Raw CMD
axes[1, 0].scatter(
    dr3["bp_rp"],
    dr3["phot_g_mean_mag"],
    s=1,
    alpha=0.2
)
axes[1, 0].invert_yaxis()
axes[1, 0].set_title("Raw DR3 CMD")
axes[1, 0].set_xlabel("BP - RP")
axes[1, 0].set_ylabel("G magnitude")

# Selected CMD
axes[1, 1].scatter(
    members_dr3["bp_rp"],
    members_dr3["phot_g_mean_mag"],
    s=3,
    alpha=0.6
)
axes[1, 1].invert_yaxis()
axes[1, 1].set_title("Selected Members CMD")
axes[1, 1].set_xlabel("BP - RP")
axes[1, 1].set_ylabel("G magnitude")

plt.tight_layout()
plt.show()


In [ ]:
print("Number of selected members:", len(members_dr3))

In [ ]:
# Compare source IDs
dr2_ids = set(dr2["source_id"])
dr3_ids = set(dr3["source_id"])

only_dr3_ids = dr3_ids - dr2_ids
only_dr2_ids = dr2_ids - dr3_ids
common_ids = dr2_ids & dr3_ids

print("Common sources:", len(common_ids))
print("Only in DR3:", len(only_dr3_ids))
print("Only in DR2:", len(only_dr2_ids))

only_dr3 = dr3[dr3["source_id"].isin(only_dr3_ids)].copy()
only_dr2 = dr2[dr2["source_id"].isin(only_dr2_ids)].copy()

print("Only DR3 dataframe:", len(only_dr3))
print("Only DR2 dataframe:", len(only_dr2))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist2d(only_dr3["ra"], only_dr3["dec"], bins=100)
axes[0].set_title("Sources only in DR3: RA-Dec")
axes[0].set_xlabel("RA")
axes[0].set_ylabel("Dec")
axes[0].invert_xaxis()

axes[1].hist2d(
    only_dr3["pmra"],
    only_dr3["pmdec"],
    bins=150,
    range=[[-5, 5], [-5, 5]]
)
axes[1].set_title("Sources only in DR3: Proper Motion")
axes[1].set_xlabel("pmRA")
axes[1].set_ylabel("pmDec")

axes[2].scatter(
    only_dr3["bp_rp"],
    only_dr3["phot_g_mean_mag"],
    s=2,
    alpha=0.4
)
axes[2].invert_yaxis()
axes[2].set_title("Sources only in DR3: CMD")
axes[2].set_xlabel("BP - RP")
axes[2].set_ylabel("G")

plt.tight_layout()
plt.show()